# Google AlloyDB for PostgreSQL - `AlloyDBRerank`

> [AlloyDB](https://cloud.google.com/alloydb) is a fully managed relational database service that offers high performance, seamless integration, and impressive scalability. AlloyDB is 100% compatible with PostgreSQL. Extend your database application to build AI-powered experiences leveraging AlloyDB's LlamaIndex integrations.

This notebook goes over how to rerank retrieved nodes with AlloyDB AI's [ranking function](https://cloud.google.com/alloydb/docs/ai/rank-rerank-search-results-rag) using the `AlloyDBRerank` node postprocessor.

Learn more about the package on [GitHub](https://github.com/googleapis/llama-index-alloydb-pg-python/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googleapis/llama-index-alloydb-pg-python/blob/main/samples/llama_index_rerank.ipynb)

## Before you begin

To run this notebook, you will need to do the following:

 * [Create a Google Cloud Project](https://developers.google.com/workspace/guides/create-project)
 * [Enable the AlloyDB API](https://console.cloud.google.com/flows/enableapi?apiid=alloydb.googleapis.com)
 * [Create a AlloyDB cluster and instance.](https://cloud.google.com/alloydb/docs/cluster-create)
 * [Create a AlloyDB database.](https://cloud.google.com/alloydb/docs/quickstart/create-and-connect)
 * [Add a User to the database.](https://cloud.google.com/alloydb/docs/database-users/about)
 * Turn on the `google_ml_integration.enable_ai_query_engine` database flag.
 * Grant the AlloyDB service agent the `roles/discoveryengine.viewer` IAM role. See [Rerank search results](https://cloud.google.com/alloydb/docs/ai/rank-rerank-search-results-rag).

### 🦙 Library Installation
Install the integration library, `llama-index-alloydb-pg`.

In [ ]:
%pip install --upgrade --quiet llama-index-alloydb-pg

**Colab only:** Uncomment the following cell to restart the kernel or use the button to restart the kernel. For Vertex AI Workbench you can restart the terminal using the button on top.

In [ ]:
# # Automatically restart kernel after installs so that your environment can access the new packages
# import IPython

# app = IPython.Application.instance()
# app.kernel.do_shutdown(True)

### 🔐 Authentication
Authenticate to Google Cloud as the IAM user logged into this notebook in order to access your Google Cloud Project.

* If you are using Colab to run this notebook, use the cell below and continue.
* If you are using Vertex AI Workbench, check out the setup instructions [here](https://github.com/GoogleCloudPlatform/generative-ai/tree/main/setup-env).

In [ ]:
from google.colab import auth

auth.authenticate_user()

### ☁ Set Your Google Cloud Project
Set your Google Cloud project so that you can leverage Google Cloud resources within this notebook.

In [ ]:
# @markdown Please fill in the value below with your Google Cloud project ID and then run the cell.

PROJECT_ID = "my-project-id"  # @param {type:"string"}

# Set the project id
!gcloud config set project {PROJECT_ID}

## Basic Usage

### Set AlloyDB database values
Find your database values, in the [AlloyDB Instances page](https://console.cloud.google.com/alloydb/clusters).

In [ ]:
# @title Set Your Values Here { display-mode: "form" }
REGION = "us-central1"  # @param {type: "string"}
CLUSTER = "my-cluster"  # @param {type: "string"}
INSTANCE = "my-primary"  # @param {type: "string"}
DATABASE = "my-database"  # @param {type: "string"}
USER = "postgres"  # @param {type: "string"}
PASSWORD = "my-password"  # @param {type: "string"}

### AlloyDBEngine Connection Pool

`AlloyDBRerank` runs the ranking in your database through an `AlloyDBEngine`. See the [`AlloyDBVectorStore` guide](./llama_index_vector_store.ipynb) for the connection options.

In [ ]:
from llama_index_alloydb_pg import AlloyDBEngine

engine = await AlloyDBEngine.afrom_instance(
    project_id=PROJECT_ID,
    region=REGION,
    cluster=CLUSTER,
    instance=INSTANCE,
    database=DATABASE,
    user=USER,
    password=PASSWORD,
)

### Rerank nodes

Create an `AlloyDBRerank` with a ranking model and the number of nodes to keep:

* `model_id`: the ranking model, for example `semantic-ranker-default-003`.
* `top_n`: the number of nodes to return. Defaults to 2.

The postprocessor returns new nodes with the rank score, highest first.

In [ ]:
from llama_index.core.schema import NodeWithScore, TextNode

from llama_index_alloydb_pg import AlloyDBRerank

nodes = [
    NodeWithScore(node=TextNode(text="Alpha: Introduction to Machine Learning.")),
    NodeWithScore(node=TextNode(text="Beta: Deep Learning and Neural Networks.")),
    NodeWithScore(node=TextNode(text="Gamma: Advanced Database Systems in Cloud.")),
]

reranker = AlloyDBRerank(engine=engine, model_id="semantic-ranker-default-003", top_n=2)
reranked = await reranker.apostprocess_nodes(nodes, query_str="neural networks")
for node in reranked:
    print(node.score, node.node.get_content())

### Use the reranker in a query engine

Pass the reranker as a node postprocessor. The retriever fetches a wide set of candidates and `AlloyDBRerank` keeps the most relevant ones. In the snippet below, `index` is a `VectorStoreIndex`, for example one built on an `AlloyDBVectorStore` as in the [`AlloyDBVectorStore` guide](./llama_index_vector_store.ipynb), and the query engine answers with the LLM set in `Settings.llm`.

```python
query_engine = index.as_query_engine(similarity_top_k=10, node_postprocessors=[reranker])
response = await query_engine.aquery("What is deep learning?")
print(response)
```